# Polymarket Insider Detection
## Detecting Statistically Significant Edge: Wallet Hit Rate vs Market-Implied Probability

**Core idea (from lecturer):** Under the efficient market null, a wallet with no private information should win at exactly the rate the market implied — i.e. their hit rate ≈ the price they paid.
A wallet that systematically wins *more* than the market predicted is a statistical fingerprint of informed trading.

We formalise this as a **binomial z-test per wallet**, integrate the result as a feature into XGBoost,
and evaluate with PR-AUC (the correct metric at ~7% class imbalance).

**Input:** `data/analytical/feature_matrix.csv` (372k rows × 40 features, pre-built)  
**Output:** ranking scores per trade, SHAP explanations, economic lift chart

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.stats import norm
import xgboost as xgb
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import CalibratedClassifierCV
import shap
import optuna

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import statsmodels.api as sm
import os

optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)

print('Libraries loaded ✓')

---
## 1 — Load Data & Define Clean Feature Set

In [ ]:
fm = pd.read_csv('feature_matrix.csv', parse_dates=['resolution_date', 'entry_date'])

print(f'Raw shape: {fm.shape}')
print(f'\nLabel distribution:')
print(fm['informed_label'].value_counts())
print(f'Positive rate: {fm["informed_label"].mean():.4f} ({fm["informed_label"].mean()*100:.1f}%)')

# Drop leaky columns: outcome_correct is part of the label definition,
# bet_outcome / wallet_direction encode the same side information
DROP_COLS = [
    'outcome_correct', 'bet_outcome', 'wallet_direction',
    'wallet', 'condition_id', 'question', 'resolution_date', 'entry_date',
    'market_volume',   # duplicated by log_market_volume
]

FEATURES = [c for c in fm.columns if c not in DROP_COLS + ['informed_label']]
print(f'\nFeatures used ({len(FEATURES)}):')
print(FEATURES)

print('\n=== Label construction audit ===')
pos = fm[fm['informed_label']==1]
print(f'Positives (informed_label=1): {len(pos):,}')
print(f'  of which outcome_correct=0: {(pos["outcome_correct"]==0).sum():,}  ← multi-trade rows where qualifying trade outcome differs from "first" trade')
print(f'  of which avg_price > 0.5:   {(pos["avg_price"] > 0.5).sum():,}  ← avg across all trades, not just the qualifying trade')
print(f'  of which hours_before <= 24: {(pos["hours_before"] <= 24).sum():,}')
print('Note: these inconsistencies arise from wallet-market aggregation, not from modelling errors.')
print('Tautological features (avg_price, hours_before) were excluded from the model for this reason.')

---
## 2 — The Lecturer's Signal: Wallet-Level Binomial Edge Test

For each wallet:
- **Expected wins** under H₀ = Σ(price paid on each bet)  
  *(if markets are efficient, a wallet with no edge wins exactly as often as the price implied)*
- **Actual wins** = how many bets resolved in their favour
- **Test statistic:** `z = (actual_wins − expected_wins) / √(Σ pᵢ(1−pᵢ))`

A wallet with z > 1.645 is winning significantly more (one-tailed test, p < 0.05)

In [ ]:
# Work from per-trade data: we need outcome_correct here for the test,
# but it will NOT enter the ML feature set
trade_level = fm[['wallet', 'avg_price', 'outcome_correct', 'informed_label', 'n_prior_bets']].copy()

# Wallet-level aggregation for the binomial test
# p_i = avg_price (implied probability the wallet paid)
wallet_stats = (
    fm.groupby('wallet')
    .agg(
        n_bets         = ('outcome_correct', 'count'),
        actual_wins    = ('outcome_correct', 'sum'),
        expected_wins  = ('avg_price', 'sum'),          # Σ p_i
        variance_wins  = ('avg_price', lambda x: (x * (1 - x)).sum()),  # Σ p_i(1-p_i)
        hit_rate       = ('outcome_correct', 'mean'),
        avg_impl_prob  = ('avg_price', 'mean'),
    )
    .reset_index()
)

# Binomial z-statistic (normal approximation, valid for n >= 10)
wallet_stats['z_stat'] = (
    (wallet_stats['actual_wins'] - wallet_stats['expected_wins'])
    / np.sqrt(wallet_stats['variance_wins'].clip(1e-8))
)
wallet_stats['p_value'] = norm.sf(wallet_stats['z_stat'])  # one-tailed: testing for positive edge
wallet_stats['edge']    = wallet_stats['hit_rate'] - wallet_stats['avg_impl_prob']

# Flag statistically significant wallets (5% threshold, one-tailed)
wallet_stats['sig_edge']    = (wallet_stats['z_stat'] > 1.645).astype(int)  # p < 0.05, one-tailed
wallet_stats['strong_edge'] = (wallet_stats['z_stat'] > 2.326).astype(int)  # p < 0.01, one-tailed

print(f'Total wallets analysed: {len(wallet_stats):,}')
print(f'Wallets with z > 1.645 (p<0.05): {wallet_stats["sig_edge"].sum():,} ({wallet_stats["sig_edge"].mean()*100:.1f}%)')
print(f'Wallets with z > 2.33  (p<0.01): {wallet_stats["strong_edge"].sum():,} ({wallet_stats["strong_edge"].mean()*100:.1f}%)')
print(f'\nTop 10 wallets by z-statistic:')
cols_show = ['wallet', 'n_bets', 'actual_wins', 'expected_wins', 'hit_rate', 'avg_impl_prob', 'edge', 'z_stat', 'p_value']
display(wallet_stats[wallet_stats['n_bets'] >= 10].nlargest(10, 'z_stat')[cols_show].round(4))

---
## 3 — Five Key Visualisations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Polymarket Insider Detection: Key Signals', fontsize=16, fontweight='bold', y=1.01)

# ── Plot 1: Z-statistic distribution with significance thresholds ──
ax = axes[0, 0]
# Use experienced wallets (>=10 bets) for cleaner distribution
exp = wallet_stats[wallet_stats['n_bets'] >= 10]
ax.hist(exp['z_stat'].clip(-5, 15), bins=60, color='steelblue', alpha=0.7, edgecolor='white', linewidth=0.3)
ax.axvline(1.645, color='orange', linestyle='--', linewidth=2, label='p<0.05 (z=1.645)')
ax.axvline(2.33,  color='red',    linestyle='--', linewidth=2, label='p<0.01 (z=2.33)')
# Overlay normal curve for reference
x_range = np.linspace(-5, 15, 200)
n_scale = len(exp) * (20 / 60)  # approximate bin width scaling
ax.plot(x_range, norm.pdf(x_range) * n_scale, 'k--', alpha=0.4, linewidth=1.5, label='H₀ (Normal)')
ax.set_xlabel('Binomial Z-statistic', fontsize=11)
ax.set_ylabel('Number of Wallets', fontsize=11)
ax.set_title('Edge Z-statistic Distribution\n(wallets with ≥10 bets)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(-5, 15)
ax.annotate(f'{exp["sig_edge"].sum()} sig. wallets\n({exp["sig_edge"].mean()*100:.1f}%)',
            xy=(0.72, 0.82), xycoords='axes fraction', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))

# ── Plot 2: Edge vs N-bets coloured by significance ──
ax = axes[0, 1]
insig = exp[exp['sig_edge'] == 0]
sig   = exp[exp['sig_edge'] == 1]
ax.scatter(insig['n_bets'].clip(0, 500), insig['edge'], alpha=0.2, s=8, color='steelblue', label='Not significant')
ax.scatter(sig['n_bets'].clip(0, 500),   sig['edge'],   alpha=0.5, s=15, color='crimson',   label='Sig. edge (p<0.05)')
ax.axhline(0, color='black', linewidth=1, linestyle='-')
ax.set_xlabel('Number of Bets (capped at 500)', fontsize=11)
ax.set_ylabel('Edge (Hit Rate − Implied Prob)', fontsize=11)
ax.set_title('Edge vs Experience\nRed = Statistically Significant', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

# ── Plot 3: Informed rate by z-score quintile ──
ax = axes[0, 2]
# Merge wallet z-stats back to trade level
fm_merged = fm.merge(wallet_stats[['wallet', 'z_stat', 'edge', 'sig_edge']], on='wallet', how='left')
fm_merged['z_quintile'] = pd.qcut(fm_merged['z_stat'], q=5, labels=['Q1\n(Lowest)', 'Q2', 'Q3', 'Q4', 'Q5\n(Highest)'])
q_rates = fm_merged.groupby('z_quintile', observed=True)['informed_label'].mean() * 100
bars = ax.bar(q_rates.index, q_rates.values,
              color=['#d73027', '#f46d43', '#fdae61', '#74add1', '#313695'],
              edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, q_rates.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xlabel('Z-statistic Quintile', fontsize=11)
ax.set_ylabel('Informed Trade Rate (%)', fontsize=11)
ax.set_title('Informed Rate by Wallet Z-score Quintile\n(Higher Z → More Informed Trades)', fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())

# ── Plot 4: Feature vs informed rate — hours_before ──
ax = axes[1, 0]
fm_tmp = fm.copy()
fm_tmp['hours_bucket'] = pd.cut(fm_tmp['hours_before'].clip(0, 500), bins=20)
hrate = fm_tmp.groupby('hours_bucket', observed=True).agg(
    informed_rate=('informed_label', 'mean'),
    count=('informed_label', 'count')
).reset_index()
# Get midpoints
hrate['mid'] = hrate['hours_bucket'].apply(lambda x: x.mid)
ax.plot(hrate['mid'], hrate['informed_rate'] * 100, 'o-', color='steelblue', markersize=5, linewidth=2)
ax.fill_between(hrate['mid'], hrate['informed_rate'] * 100, alpha=0.15, color='steelblue')
ax.set_xlabel('Hours Before Resolution', fontsize=11)
ax.set_ylabel('Informed Trade Rate (%)', fontsize=11)
ax.set_title('Timing vs Informed Rate\n(Non-linear → justifies XGBoost over logistic)', fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.axvline(24, color='red', linestyle='--', alpha=0.5, label='24h cutoff')
ax.legend(fontsize=9)

# ── Plot 5: log_net_usd vs informed rate ──
ax = axes[1, 1]
fm_tmp['usd_bucket'] = pd.cut(fm_tmp['log_net_usd'], bins=20)
urate = fm_tmp.groupby('usd_bucket', observed=True).agg(
    informed_rate=('informed_label', 'mean'),
    count=('informed_label', 'count')
).reset_index()
urate['mid'] = urate['usd_bucket'].apply(lambda x: x.mid)
ax.plot(urate['mid'], urate['informed_rate'] * 100, 's-', color='darkorange', markersize=5, linewidth=2)
ax.fill_between(urate['mid'], urate['informed_rate'] * 100, alpha=0.15, color='darkorange')
ax.set_xlabel('Log Net USD Position', fontsize=11)
ax.set_ylabel('Informed Trade Rate (%)', fontsize=11)
ax.set_title('Position Size vs Informed Rate\n(Larger bets → much higher informed rate)', fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())

# ── Plot 6: Correlation heatmap of top features vs label ──
ax = axes[1, 2]
feat_corr_cols = [
    'z_score_roll', 'log_net_usd', 'hit_rate_pct', 'raw_edge_roll',
    'hours_before', 'avg_price', 'divergence_score',
    '^VIX_7d', 'ft_sentiment_score_z', 'n_prior_bets'
]
corrs = fm[feat_corr_cols + ['informed_label']].corr()['informed_label'].drop('informed_label').sort_values()
colors = ['crimson' if v < 0 else 'steelblue' for v in corrs.values]
ax.barh(corrs.index, corrs.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with Informed Label', fontsize=11)
ax.set_title('Feature Correlations with Target\n(z_score_roll dominates)', fontsize=12, fontweight='bold')
for i, (val, idx) in enumerate(zip(corrs.values, corrs.index)):
    ax.text(val + (0.003 if val >= 0 else -0.003), i, f'{val:.3f}',
            va='center', ha='left' if val >= 0 else 'right', fontsize=9)

plt.tight_layout()
#plt.savefig('figures/key_signals.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/key_signals.png')

---
## 4 — Feature Engineering: Add Wallet Z-statistic to Feature Matrix

In [ ]:
os.makedirs('figures', exist_ok=True)

# Register the new feature names — the actual values will be merged in
# AFTER the temporal split (below), so test bets don't contaminate training stats
NEW_FEATURES = ['wallet_z_stat', 'wallet_edge', 'wallet_sig_edge', 'wallet_strong_edge']
FEATURES = FEATURES + NEW_FEATURES
# These three features directly encode label conditions — remove them
TAUTOLOGICAL = ['avg_price', 'log_net_usd', 'hours_before','hit_rate_pct']
FEATURES = [f for f in FEATURES if f not in TAUTOLOGICAL]
 
print(f'Feature set now has {len(FEATURES)} features')
print(f'New features to be added post-split: {NEW_FEATURES}')

---
## 5 — Temporal Train/Test Split (No Leakage)

**Critical:** We split on `resolution_date`, not randomly. Markets resolved before the 70th percentile date form the training set; later markets are held out. This mirrors real-world deployment where you train on historical data and predict on future markets.

In [ ]:
# Temporal split: 70% train, 30% test by resolution date
# resolution_date may load as timezone-aware strings — convert explicitly
fm['resolution_date'] = pd.to_datetime(fm['resolution_date'],format='ISO8601', utc=True)

# Sort by date, then use iloc position rather than .quantile() (avoids interpolation on timestamps)
fm_sorted = fm.sort_values('resolution_date').reset_index(drop=True)
cutoff_idx = int(len(fm_sorted) * 0.70)
cutoff_date = fm_sorted['resolution_date'].iloc[cutoff_idx]
print(f'Train cutoff: {cutoff_date}')

train = fm_sorted.iloc[:cutoff_idx].copy()
test  = fm_sorted.iloc[cutoff_idx:].copy()

# Fill any NaN in features (can appear in rolling features for early trades)
for df in [train, test]:
    for col in FEATURES:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

# Build arrays, ensuring all features exist
FEATURES_FINAL = [f for f in FEATURES if f in fm.columns]

X_train = train[FEATURES_FINAL]
y_train = train['informed_label']
X_test  = test[FEATURES_FINAL]
y_test  = test['informed_label']

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# ── Compute wallet z-stats from TRAINING data only (no look-ahead) ──
train_wallet_stats = (
    train.groupby('wallet')
    .agg(
        n_bets        = ('outcome_correct', 'count'),
        actual_wins   = ('outcome_correct', 'sum'),
        expected_wins = ('avg_price', 'sum'),
        variance_wins = ('avg_price', lambda x: (x * (1 - x)).sum()),
        hit_rate      = ('outcome_correct', 'mean'),
        avg_impl_prob = ('avg_price', 'mean'),
    )
    .reset_index()
)
train_wallet_stats['wallet_z_stat']    = (
    (train_wallet_stats['actual_wins'] - train_wallet_stats['expected_wins'])
    / np.sqrt(train_wallet_stats['variance_wins'].clip(1e-8))
)
train_wallet_stats['wallet_edge']      = train_wallet_stats['hit_rate'] - train_wallet_stats['avg_impl_prob']
train_wallet_stats['wallet_sig_edge']  = (train_wallet_stats['wallet_z_stat'] > 1.645).astype(int)
train_wallet_stats['wallet_strong_edge'] = (train_wallet_stats['wallet_z_stat'] > 2.33).astype(int)

merge_cols = ['wallet', 'wallet_z_stat', 'wallet_edge', 'wallet_sig_edge', 'wallet_strong_edge']

# Merge into both splits — test wallets get stats derived only from training history
train = train.merge(train_wallet_stats[merge_cols], on='wallet', how='left')
test  = test.merge(train_wallet_stats[merge_cols],  on='wallet', how='left')

# Rebuild X arrays now that the new columns exist
FEATURES_FINAL = [f for f in FEATURES if f in train.columns]
X_train = train[FEATURES_FINAL].fillna(0)
X_test  = test[FEATURES_FINAL].fillna(0)

print(f'\nTrain: {len(train):,} rows | Test: {len(test):,} rows')
print(f'Train positive rate: {y_train.mean():.4f}')
print(f'Test positive rate:  {y_test.mean():.4f}')
print(f'scale_pos_weight:    {pos_weight:.1f}')

---
## 6 — Logistic Regression Baseline

Always benchmark against logistic regression before deploying a complex model. If XGBoost doesn't beat logistic by a meaningful margin on PR-AUC, the non-linearity justification falls away.

In [ ]:
# Logistic needs scaling; XGBoost does not
baseline_features = [
    #'log_net_usd', 'avg_price', 'hours_before', 
    #'hit_rate_pct',
    'z_score_roll', 'divergence_score', 'wallet_z_stat', 'wallet_edge'
]
baseline_features = [f for f in baseline_features if f in X_train.columns]

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(class_weight='balanced', max_iter=500, random_state=SEED))
])
pipe.fit(X_train[baseline_features], y_train)
lr_probs = pipe.predict_proba(X_test[baseline_features])[:, 1]

lr_roc  = roc_auc_score(y_test, lr_probs)
lr_pr   = average_precision_score(y_test, lr_probs)

print(f'Logistic Regression Baseline:')
print(f'  ROC-AUC: {lr_roc:.4f}')
print(f'  PR-AUC:  {lr_pr:.4f}')

In [ ]:
# Statsmodels needs scaled data (same as the sklearn pipeline) and a constant term
scaler_sm = StandardScaler()
X_train_sm = scaler_sm.fit_transform(X_train[baseline_features])
X_test_sm  = scaler_sm.transform(X_test[baseline_features])

# Add intercept column (statsmodels doesn't add it automatically)
X_train_sm_c = sm.add_constant(X_train_sm)
X_test_sm_c  = sm.add_constant(X_test_sm)

# Fit statsmodels logit
sm_logit = sm.Logit(y_train, X_train_sm_c)
sm_result = sm_logit.fit(maxiter=300, disp=False)

# Swap out the generic 'x1, x2...' names for your actual feature names
sm_result.model.exog_names[:] = ['const'] + baseline_features

print(sm_result.summary())

---
## 7 — XGBoost with Optuna Tuning

**Objective: PR-AUC** — the correct metric for imbalanced classification (7% positive rate).
ROC-AUC is reported alongside for completeness but PR-AUC drives all decisions.

In [ ]:
# Quick baseline XGBoost before tuning
xgb_base = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=pos_weight,
    eval_metric='aucpr',
    use_label_encoder=False,
    random_state=SEED,
    n_jobs=-1
)
xgb_base.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

base_probs = xgb_base.predict_proba(X_test)[:, 1]
base_roc   = roc_auc_score(y_test, base_probs)
base_pr    = average_precision_score(y_test, base_probs)

print(f'XGBoost (baseline, untuned):')
print(f'  ROC-AUC: {base_roc:.4f}')
print(f'  PR-AUC:  {base_pr:.4f}')
print(f'  Lift over logistic PR-AUC: +{base_pr - lr_pr:.4f}')

In [ ]:

# Optuna tuning — optimising mean CV PR-AUC on the training set only
X_train_cv = X_train.reset_index(drop=True)
y_train_cv = y_train.reset_index(drop=True)
cv_splitter = TimeSeriesSplit(n_splits=4)

def objective(trial):
    params = {
        'max_depth':           trial.suggest_int('max_depth', 2, 8),
        'learning_rate':       trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators':        trial.suggest_int('n_estimators', 100, 600),
        'min_child_weight':    trial.suggest_int('min_child_weight', 1, 10),
        'subsample':           trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':    trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha':           trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda':          trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'scale_pos_weight':    trial.suggest_float('scale_pos_weight', 5.0, 30.0),
        'eval_metric':         'aucpr',
        'use_label_encoder':   False,
        'random_state':        SEED,
        'n_jobs':              -1,
    }

    fold_scores = []
    for fit_idx, val_idx in cv_splitter.split(X_train_cv):
        X_fit = X_train_cv.iloc[fit_idx]
        y_fit = y_train_cv.iloc[fit_idx]
        X_val = X_train_cv.iloc[val_idx]
        y_val = y_train_cv.iloc[val_idx]

        model = xgb.XGBClassifier(**params)
        model.fit(X_fit, y_fit, verbose=False)
        preds = model.predict_proba(X_val)[:, 1]
        fold_scores.append(average_precision_score(y_val, preds))

    return float(np.mean(fold_scores))

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=60, show_progress_bar=True)

print(f'\nBest mean CV PR-AUC: {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

In [ ]:
# Retrain final model on full training set, then evaluate once on held-out test
best_params = study.best_params.copy()
best_params.update({'eval_metric': 'aucpr', 'use_label_encoder': False, 
                    'random_state': SEED, 'n_jobs': -1})

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train, verbose=False)

final_train_probs = final_model.predict_proba(X_train)[:, 1]
final_probs = final_model.predict_proba(X_test)[:, 1]
final_train_pr = average_precision_score(y_train, final_train_probs)
final_roc = roc_auc_score(y_test, final_probs)
final_pr = average_precision_score(y_test, final_probs)

# Sweep max_depth as a simple proxy for model flexibility
flex_rows = []
for depth in range(1, 9):
    flex_params = best_params.copy()
    flex_params['max_depth'] = depth
    flex_model = xgb.XGBClassifier(**flex_params)
    flex_model.fit(X_train, y_train, verbose=False)

    train_pr_depth = average_precision_score(y_train, flex_model.predict_proba(X_train)[:, 1])
    test_pr_depth = average_precision_score(y_test, flex_model.predict_proba(X_test)[:, 1])
    flex_rows.append({
        'max_depth': depth,
        'train_pr_auc': train_pr_depth,
        'test_pr_auc': test_pr_depth,
    })

flex_df = pd.DataFrame(flex_rows)

print('=== FINAL MODEL RESULTS ===')
print(f'  Mean CV PR-AUC: {study.best_value:.4f}')
print(f'  Train PR-AUC:   {final_train_pr:.4f}')
print(f'  Test ROC-AUC:   {final_roc:.4f}')
print(f'  Test PR-AUC:    {final_pr:.4f}')
print(f'  Lift over logistic: +{final_pr - lr_pr:.4f}')
print(f'  Lift over XGB base: +{final_pr - base_pr:.4f}')

In [ ]:
# # Apply Platt scaling (sigmoid) calibration on a held-out portion of training data
# # Use cv='prefit' since model is already fitted
# calibrated_model = CalibratedClassifierCV(final_model, cv='prefit', method='sigmoid')
# calibrated_model.fit(X_train, y_train)

# # Replace raw probabilities with calibrated ones for all downstream use
# final_probs_raw = final_probs.copy()          # keep raw for comparison
# final_probs = calibrated_model.predict_proba(X_test)[:, 1]

# # Verify calibration improved
# mean_pred_raw  = final_probs_raw.mean()
# mean_pred_cal  = final_probs.mean()
# actual_rate    = y_test.mean()
# print(f'Actual positive rate:           {actual_rate:.4f}')
# print(f'Mean predicted prob (raw):      {mean_pred_raw:.4f}')
# print(f'Mean predicted prob (calibrated): {mean_pred_cal:.4f}')

# # Recalculate final_pr with calibrated probs
# final_pr  = average_precision_score(y_test, final_probs)
# final_roc = roc_auc_score(y_test, final_probs)
# print(f'\nCalibrated model — Test PR-AUC: {final_pr:.4f}  ROC-AUC: {final_roc:.4f}')

In [ ]:
# ── Ablation: does the three-way divergence story hold up? ──────────────────
wallet_feats     = [f for f in FEATURES_FINAL if f in [
    'z_score_roll','raw_edge_roll','n_prior_bets',
    'wallet_z_stat','wallet_edge','wallet_sig_edge','wallet_strong_edge']]
non_wallet_feats = [f for f in FEATURES_FINAL if f not in wallet_feats]

ablation_results = {}
for label, feat_set in [
    ('Wallet-history only',   wallet_feats),
    ('Divergence/sentiment only', non_wallet_feats),
    ('Full feature set (30)', FEATURES_FINAL),
]:
    m = xgb.XGBClassifier(**best_params)
    m.fit(X_train[feat_set], y_train, verbose=False)
    pr = average_precision_score(y_test, m.predict_proba(X_test[feat_set])[:,1])
    ablation_results[label] = {'N features': len(feat_set), 'Test PR-AUC': round(pr, 4)}

ablation_df = pd.DataFrame(ablation_results).T
display(ablation_df)
print()
print('INTERPRETATION:')
print('  If wallet-history alone achieves comparable PR-AUC to the full set, the model is')
print('  primarily a skill-persistence classifier. Divergence/sentiment signals provide')
print('  secondary context, not independent discriminative power.')

# Fair logistic comparison: use ALL features, not just 4
pipe_full = Pipeline([('sc', StandardScaler()),
                      ('lr', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED))])
pipe_full.fit(X_train, y_train)
lr_full_pr = average_precision_score(y_test, pipe_full.predict_proba(X_test)[:,1])
print(f'\nFair logistic baseline (all {len(FEATURES_FINAL)} features): PR-AUC = {lr_full_pr:.4f}')
print(f'XGBoost (all {len(FEATURES_FINAL)} features):              PR-AUC = {final_pr:.4f}')
print(f'True XGBoost lift over equivalent logistic: +{final_pr - lr_full_pr:.4f}')

---
## 8 — Model Evaluation: ROC, PR, Flexibility, and Lift

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Evaluation', fontsize=14, fontweight='bold')

# ── ROC Curve ──
ax = axes[0, 0]
for probs, label, color in [
    (lr_probs,    f'Logistic (AUC={lr_roc:.3f})', 'steelblue'),
    (base_probs,  f'XGB base (AUC={base_roc:.3f})', 'darkorange'),
    (final_probs, f'XGB tuned (AUC={final_roc:.3f})', 'crimson'),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax.plot(fpr, tpr, label=label, linewidth=2, color=color)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, linewidth=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── PR Curve ──
ax = axes[0, 1]
baseline_precision = y_test.mean()
ax.axhline(baseline_precision, color='gray', linestyle='--', linewidth=1,
           label=f'No-skill baseline ({baseline_precision:.3f})')
for probs, label, color in [
    (lr_probs,    f'Logistic (PR-AUC={lr_pr:.3f})', 'steelblue'),
    (base_probs,  f'XGB base (PR-AUC={base_pr:.3f})', 'darkorange'),
    (final_probs, f'XGB tuned (PR-AUC={final_pr:.3f})', 'crimson'),
]:
    prec, rec, _ = precision_recall_curve(y_test, probs)
    ax.plot(rec, prec, label=label, linewidth=2, color=color)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve\n(primary metric at 7% imbalance)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── PR-AUC vs model flexibility ──
ax = axes[1, 0]
ax.plot(flex_df['max_depth'], flex_df['train_pr_auc'], 'o-', color='steelblue', linewidth=2,
        label='Train PR-AUC')
ax.plot(flex_df['max_depth'], flex_df['test_pr_auc'], 'o-', color='crimson', linewidth=2,
        label='Test PR-AUC')
best_depth = int(flex_df.loc[flex_df['test_pr_auc'].idxmax(), 'max_depth'])
ax.axvline(best_depth, color='gray', linestyle='--', linewidth=1,
           label=f'Best test depth = {best_depth}')
ax.set_xlabel('Max Depth (proxy for model flexibility)')
ax.set_ylabel('PR-AUC')
ax.set_title('PR-AUC vs Model Flexibility\n(train vs test across tree depth)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── Cumulative Lift Chart ──
ax = axes[1, 1]
test_scores = test[['informed_label']].copy()
test_scores['prob'] = final_probs
test_scores = test_scores.sort_values('prob', ascending=False).reset_index(drop=True)
n_test = len(test_scores)
base_rate = y_test.mean()

depths = np.arange(1, n_test + 1) / n_test * 100
cum_precision = test_scores['informed_label'].cumsum() / np.arange(1, n_test + 1)
lift = cum_precision / base_rate

ax.plot(depths, lift, color='crimson', linewidth=2)
ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, label='Baseline (lift=1)')
ax.axvline(10, color='steelblue', linestyle=':', linewidth=1.5, alpha=0.7, label='Top 10%')
ax.axvline(20, color='darkorange', linestyle=':', linewidth=1.5, alpha=0.7, label='Top 20%')
idx_10 = max(int(np.ceil(n_test * 0.10)) - 1, 0)
lift_10 = lift.iloc[idx_10]
ax.annotate(f'{lift_10:.1f}x lift\nat top 10%', xy=(10, lift_10),
            xytext=(20, lift_10 + 0.3),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=11, fontweight='bold')
ax.set_xlabel('% of Trades Flagged (by model score, desc)', fontsize=11)
ax.set_ylabel('Lift over Baseline', fontsize=11)
ax.set_title('Cumulative Lift Chart\n(how much better than random?)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9 — SHAP Explainability

In [ ]:
# SHAP on a random subsample of test set (full set is slow)
np.random.seed(SEED)
shap_idx = np.random.choice(len(X_test), size=min(5000, len(X_test)), replace=False)
X_shap = X_test.iloc[shap_idx]

explainer   = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_shap)

fig, axes = plt.subplots(1, 2, figsize=(16, 22))

# Bar plot (mean |SHAP|)
plt.sca(axes[0])
shap.summary_plot(shap_values, X_shap, plot_type='bar', max_display=15, show=False)
axes[0].set_title('SHAP Feature Importance\n(Mean |SHAP value|)', fontweight='bold', fontsize=12)

# Beeswarm
plt.sca(axes[1])
shap.summary_plot(shap_values, X_shap, max_display=15, show=False)
axes[1].set_title('SHAP Beeswarm\n(Feature effect direction)', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('figures/shap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10 — The Lecturer's Signal: Economic Validation

The model predicts *which* trades come from informed wallets. If the signal is real:
- Trades the model flags should have higher actual hit rates
- z > 1.645 (one-tailed, p < 0.05) so high-z wallets lose less out-of-sample — though no group achieves positive edge in an efficient market

In [ ]:
test_eval = test.copy()

test_eval['model_score'] = final_probs  # re-assign calibrated scores

print('=== ECONOMIC VALIDATION ===')
print(f'Baseline: outcome hit rate = {test["outcome_correct"].mean():.3f} | informed-label rate = {test["informed_label"].mean():.3f}')
print()

top_fracs = [0.50, 0.40, 0.30, 0.20, 0.10]
results = []
for top_frac in top_fracs:
    thresh   = np.quantile(final_probs, 1 - top_frac)
    flagged  = test_eval[test_eval['model_score'] >= thresh]
    if len(flagged) == 0: continue
    hit_rate = flagged['outcome_correct'].mean()
    pr       = flagged['informed_label'].mean()
    outcome_edge = hit_rate - test['outcome_correct'].mean()
    results.append({
        'Top % flagged':      f'Top {int(top_frac*100)}%',
        'N trades':           len(flagged),
        'Outcome hit rate':   f'{hit_rate:.3f}',
        'vs baseline (+pp)':  f'{outcome_edge*100:+.2f}pp',
        'Label precision':    f'{pr:.3f}',
        'Label lift':         f'{pr / test["informed_label"].mean():.2f}x'
    })

results_df = pd.DataFrame(results)
display(results_df)
print()
print('INTERPRETATION:')
print('  "Label lift" = how much better than random at finding rows matching our informed_label definition.')
print('  "vs baseline" = actual out-of-sample outcome improvement — modest by design in an efficient market.')
print('  The model is a strong ranker of the informed-label pattern; it is not a market-beating signal.')

i was confused why 

In [ ]:
# The Lecturer's Signal: wallet z-stat vs subsequent outcome hit rate
# Plotting both raw hit rate AND edge (hit rate − implied prob) to remove favourite-betting distortion

test_wallet_perf = (
    test
    .merge(train_wallet_stats[['wallet', 'wallet_z_stat', 'wallet_sig_edge', 'wallet_strong_edge',
                                'avg_impl_prob']].rename(
        columns={'wallet_z_stat': 'z_stat', 'wallet_sig_edge': 'sig_edge',
                 'wallet_strong_edge': 'strong_edge', 'avg_impl_prob': 'train_avg_price'}
    ), on='wallet', how='left')
    .groupby('wallet')
    .agg(
        z_stat          = ('z_stat', 'first'),
        sig_edge        = ('sig_edge', 'first'),
        strong_edge     = ('strong_edge', 'first'),
        train_avg_price = ('train_avg_price', 'first'),
        test_hit_rate   = ('outcome_correct', 'mean'),
        test_avg_price  = ('avg_price', 'mean'),   # implied prob in test bets
        test_n          = ('outcome_correct', 'count')
    )
    .dropna()
)

# Edge = hit rate minus the implied probability the wallet paid in test
test_wallet_perf['test_edge'] = test_wallet_perf['test_hit_rate'] - test_wallet_perf['test_avg_price']

test_wallet_perf['z_decile'] = pd.qcut(test_wallet_perf['z_stat'], q=10,
                                        labels=[f'D{i}' for i in range(1, 11)])

decile_perf = test_wallet_perf.groupby('z_decile', observed=True).agg(
    test_hit_rate = ('test_hit_rate', 'mean'),
    test_edge     = ('test_edge', 'mean'),
    avg_price     = ('test_avg_price', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Lecturer's Signal: Out-of-Sample Wallet Performance by Z-stat Decile",
             fontsize=14, fontweight='bold')

colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, 10))

# ── Left: Raw hit rate (as before) ──
ax = axes[0]
bars = ax.bar(decile_perf['z_decile'], decile_perf['test_hit_rate'] * 100,
              color=colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, decile_perf['test_hit_rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9)
ax.axhline(test['outcome_correct'].mean() * 100, color='black', linestyle='--',
           linewidth=2, label=f'Avg hit rate ({test["outcome_correct"].mean()*100:.1f}%)')
ax.set_xlabel('Z-statistic Decile (D1=Lowest, D10=Highest)', fontsize=11)
ax.set_ylabel('Out-of-Sample Hit Rate (%)', fontsize=11)
ax.set_title('Raw Hit Rate\n(distorted by favourite-betting)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(axis='y', alpha=0.3)

# ── Right: Edge = hit rate − implied probability ──
ax = axes[1]
edge_colors = ['crimson' if v < 0 else c for v, c in zip(decile_perf['test_edge'], colors)]
bars = ax.bar(decile_perf['z_decile'], decile_perf['test_edge'] * 100,
              color=edge_colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, decile_perf['test_edge']):
    offset = 0.3 if val >= 0 else -1.2
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + offset,
            f'{val*100:+.1f}%', ha='center', va='bottom', fontsize=9)
ax.axhline(0, color='black', linestyle='--', linewidth=2, label='Zero edge (market is right)')
ax.set_xlabel('Z-statistic Decile (D1=Lowest, D10=Highest)', fontsize=11)
ax.set_ylabel('Out-of-Sample Edge: Hit Rate − Implied Prob (%)', fontsize=11)
ax.set_title('True Edge (Hit Rate − Implied Probability)\n(removes favourite-betting distortion)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
#plt.savefig('lecturer_signal_validation.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
low_z_edge  = test_wallet_perf[test_wallet_perf['z_decile'].isin(['D1','D2','D3'])]['test_edge'].mean()
high_z_edge = test_wallet_perf[test_wallet_perf['z_decile'].isin(['D8','D9','D10'])]['test_edge'].mean()
from scipy import stats as scipy_stats
t_stat, p_val = scipy_stats.ttest_1samp(
    test_wallet_perf[test_wallet_perf['z_decile'].isin(['D8','D9','D10'])]['test_edge'], 0)

print(f"Bottom 30% z-stat wallets — out-of-sample edge: {low_z_edge*100:+.2f}pp")
print(f"Top    30% z-stat wallets — out-of-sample edge: {high_z_edge*100:+.2f}pp")
print(f"Edge spread (top vs bottom 30%): {(high_z_edge - low_z_edge)*100:.1f}pp")
print()
print("NOTE: All deciles show negative out-of-sample edge — wallets lose against market-implied")
print("probabilities on average. High-z wallets lose significantly less (6.2pp gap), consistent")
print("with skill persistence, but the market is efficient enough that no group achieves positive edge.")
print(f"One-sample t-test (top-30% edge vs zero): t={t_stat:.3f}, p={p_val:.4f}")
print("Interpretation: the z-stat successfully ranks wallets by skill, but does not identify")
print("a group with exploitable positive edge in the test period.")

---
## 11 — Summary

| Metric | Logistic Baseline | XGB (untuned) | XGB (Optuna) |
|--------|:-----------------:|:-------------:|:------------:|
| ROC-AUC | .9226 | .9762 | .9910 |
| PR-AUC  | .5179 | .7597 | .8614 |

*(Values filled in at runtime above)*

### Key findings
1. **The lecturer's binomial z-test works.** High-z wallets lose significantly less out-of-sample (−1.4pp vs −7.6pp for bottom tercile, t=−3.517, p=0.0004), but no decile achieves positive edge — consistent with a semi-efficient geopolitical prediction market

2. **`z_score_roll` is the single strongest feature** (correlation 0.40 with informed label, SHAP dominant). It encodes the rolling version of the binomial test — trades from wallets with a persistent statistical edge are the strongest insider signal.

3. **XGBoost justifiably outperforms logistic regression.** The non-linear `hours_before` relationship (U-shaped informed rate) and interaction between position size and timing cannot be captured by a linear model without manual feature engineering.

4. **PR-AUC, not ROC-AUC, is the right metric.** At 7% positives, ROC-AUC is inflated by correct negative predictions. PR-AUC measures what matters: how precisely you identify the informed trades when you flag something.

5. **Use probability outputs, not hard labels.** The model's `predict_proba` scores allow threshold selection (e.g. flag only top-10% — high precision — vs top-30% — higher recall) and can be used directly as a continuous signal in the backtest.

Logistic Regression Baseline:

  ROC-AUC: 0.9226

  PR-AUC:  0.5179

--------------------------------------

``XGBoost (baseline, untuned):``

  ROC-AUC: 0.9762

  PR-AUC:  0.7597

--------------------------------------

  Mean CV PR-AUC: 0.7741
  
  Train PR-AUC:   0.9778
  
  Test ROC-AUC:   0.9910
  
  Test PR-AUC:    0.8614